# Open Library｜作家・作品・版の記録を調べる
**技術講習会「文学研究のためのデジタル資料活用」｜2026年10月16日**

Open Libraryでは、作家の記録を**Author**、一つの作品をまとめる記録を**Work**、個々の出版物の記録を**Edition**と呼びます。

ここでは、**一つのWorkに何件のEditionが登録されているか**を調べます。これは出版部数や読者数ではなく、Open Libraryに記録されている版の件数です。

検索には、2026年2月28日版のOpen Libraryデータを保存した講習会用サーバーを使用します。サーバーに接続できない場合は、Joseph Conradの8作品について保存済みの例を表示します。

## 1. 作家と作品を探す
作家名と作品名を入力して、実行ボタン（▶）を押してください。最初はJoseph Conradの *Heart of Darkness* を例にしています。検索するとAuthor ID（作家の記録番号）とWork ID（作品の記録番号）が表示されます。

同じ名前の別人や、同じタイトルの別作品も含まれる場合があります。**作品の検索結果は著者を指定せずに探した候補一覧**なので、次の比較でAuthor IDとの対応を確認します。

In [ ]:
# @title 作家名と作品名を入力して検索する {display-mode: "form"}
AUTHOR_NAME = "Joseph Conrad" # @param {type:"string"}
WORK_TITLE = "Heart of Darkness" # @param {type:"string"}

# 以下は変更不要です
API_URL = "https://bunsekisan.jp/ol-workshop-api"

import time
import unicodedata
import requests
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 通信できない場合だけ使う8作品の例（Open Library公開ページ、2026-10-14確認）。
# 2026-02-28版の講習会用データベースの数値とは異なる場合があります。
CONRAD_EIGHT = [
    {"work_id":"OL38630W", "title":"Almayer's Folly", "n":97, "year":1895},
    {"work_id":"OL38815W", "title":'The Nigger of the "Narcissus"', "n":93, "year":1897},
    {"work_id":"OL38663W", "title":"Heart of Darkness", "n":553, "year":1899},
    {"work_id":"OL38684W", "title":"Lord Jim", "n":345, "year":1900},
    {"work_id":"OL38700W", "title":"Nostromo", "n":255, "year":1904},
    {"work_id":"OL39108W", "title":"The Secret Agent", "n":335, "year":1907},
    {"work_id":"OL38763W", "title":"Victory", "n":146, "year":1915},
    {"work_id":"OL38827W", "title":"The Shadow-Line", "n":147, "year":1917},
]

def norm(s):
    return unicodedata.normalize("NFKC", str(s or "")).casefold().strip()

def api_get(path, params):
    # 混雑時だけ一度再試行。連続リクエストは避けます。
    for attempt in range(2):
        try:
            r = requests.get(API_URL + path, params=params, timeout=(5, 18))
            if r.status_code in (429, 503) and attempt == 0:
                time.sleep(3)
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if attempt == 1:
                raise
            time.sleep(3)
    raise RuntimeError("APIへの接続に失敗しました")

_api_available = True
try:
    a = api_get("/v1/authors", {"name": AUTHOR_NAME, "limit": 12})
    w = api_get("/v1/works", {"title": WORK_TITLE, "limit": 30})
    authors = a.get("candidates", [])
    works = w.get("works", [])
    works.sort(key=lambda x: (norm(x.get("title")) != norm(WORK_TITLE),
                              -int(x.get("linked_edition_records", 0))))
    print("検索データ：Open Library 2026年2月28日版")
except (requests.RequestException, RuntimeError) as e:
    _api_available = False
    print("講習会用サーバーに接続できませんでした。", str(e))
    if norm(AUTHOR_NAME) == "joseph conrad":
        print("代わりに、Joseph Conradの保存済み8作品から表示します。")
        authors = [{"author_id":"OL19441A", "name":"Joseph Conrad", "linked_work_records":None}]
        works = [{"work_id":x["work_id"], "title":x["title"],
                  "linked_edition_records":x["n"]} for x in CONRAD_EIGHT
                 if norm(WORK_TITLE) in norm(x["title"])]
    else:
        authors, works = [], []
        print("保存済みの例はJoseph Conradのみです。別の作家を探す場合は、接続が戻ってから再実行してください。")

print("\n【作家の候補】")
if authors:
    display(pd.DataFrame([{
        "Author ID": x["author_id"], "登録名": x["name"],
        "関連Work件数": x.get("linked_work_records", "—"),
        "Open Library": f'https://openlibrary.org/authors/{x["author_id"]}'
    } for x in authors]).head(12))
else:
    print("該当する候補はありません。")

print("\n【作品の候補】（著者による絞り込みはしていません）")
if works:
    display(pd.DataFrame([{
        "Work ID": x["work_id"], "タイトル": x["title"],
        "Edition登録件数": x.get("linked_edition_records", 0),
        "Open Library": f'https://openlibrary.org/works/{x["work_id"]}'
    } for x in works]).head(15))
else:
    print("該当する候補はありません。")

## 2. 作家・作品・年代を指定する
上の一覧で確認した**Author ID**と**Work ID**を入力します。最初はConradと *Heart of Darkness* を設定しています。

年代は空欄でも構いません。指定した場合は**その年代に対応すると登録されたWorkだけ**を比較します。ただし、Open Libraryの年は実際の初刊年と異なることがあります。また、Edition登録件数は**その年代に出版された版だけの数ではなく、各Workに結びつくすべてのEditionの登録件数**です。

In [ ]:
# @title 比較する作家・作品・年代を指定する {display-mode: "form"}
AUTHOR_ID = "OL19441A" # @param {type:"string"}
WORK_ID = "OL38663W" # @param {type:"string"}
YEAR_FROM = "" # @param {type:"string"}
YEAR_TO = "" # @param {type:"string"}
DISPLAY_LIMIT = 15 # @param {type:"integer"}

if not (1 <= int(DISPLAY_LIMIT) <= 50):
    raise ValueError("表示件数は1〜50にしてください。")
if bool(YEAR_FROM.strip()) != bool(YEAR_TO.strip()):
    raise ValueError("年代を絞る場合は、開始年と終了年の両方を入力してください。")
if YEAR_FROM.strip() and (not YEAR_FROM.isdecimal() or not YEAR_TO.isdecimal()
                          or int(YEAR_FROM) > int(YEAR_TO)):
    raise ValueError("年代を数字で正しく指定してください。")

_live_result = None
_is_saved = not _api_available
if not _is_saved:
    try:
        params = {"author_ids": AUTHOR_ID.strip(), "target_work_id": WORK_ID.strip(),
                  "display_limit": 100 if AUTHOR_ID.strip() == "OL19441A" else DISPLAY_LIMIT}
        if YEAR_FROM.strip():
            params.update({"year_from": int(YEAR_FROM), "year_to": int(YEAR_TO)})
        _live_result = api_get("/v1/collections/by-authors", params)
    except (requests.RequestException, RuntimeError) as e:
        print("検索サーバーに接続できませんでした：", e)
        _is_saved = True

if _is_saved:
    if AUTHOR_ID.strip() != "OL19441A":
        raise RuntimeError("保存済みのデータはConradの8作品だけです。ほかの作家はAPIの復旧後に検索してください。")
    rows = [x for x in CONRAD_EIGHT if not YEAR_FROM.strip()
            or int(YEAR_FROM) <= x["year"] <= int(YEAR_TO)]
    graph_rows = [{"work_id":x["work_id"], "title":x["title"],
                   "linked_edition_records":x["n"], "year":x["year"]} for x in rows]
    graph_rows = graph_rows[:DISPLAY_LIMIT]
    print("【保存済みの例】Conradの8作品。公開ページのEdition表示件数（2026年10月14日確認）")
    print("この8作品は事前に選んだ例であり、Conradの全Workの順位や中央値は計算していません。")
else:
    s = _live_result["statistics_full_matching_set"]
    print("【2026年2月28日版 Open Library】")
    print(f'比較対象：{s["work_records"]:,} Workレコード')
    print(f'Edition登録数の中央値：{s["median_linked_edition_records"]} 件')
    if s.get("target"):
        t = s["target"]
        print(f'指定作品：{t["title"]}／Edition登録数 {t["linked_edition_records"]:,}件')
        print(f'登録Work単位の順位：{t["rank_competition"]:,}位／{t["rank_denominator_work_records"]:,}件（同数同順位）')
    elif WORK_ID.strip():
        print("注意：指定したWork IDはこの作家・年代の比較集合に含まれていません。")
    items = _live_result.get("works_displayed", [])
    graph_rows = []
    if AUTHOR_ID.strip() == "OL19441A" and not YEAR_FROM.strip():
        wanted = {x["work_id"]:x for x in CONRAD_EIGHT}
        found = {x["work_id"]:x for x in items}
        for wid, meta in wanted.items():
            if wid in found:
                graph_rows.append({**found[wid], "year":meta["year"]})
        graph_rows.sort(key=lambda x: (x["year"], x["title"]))
        print("表示：確認済みのConrad 8 Workを初出・初刊年順に並べます。")
        missing = set(wanted) - set(found)
        if missing:
            print("一覧の上位100件にないWorkは省略しました：", ", ".join(sorted(missing)))
    else:
        graph_rows = items[:DISPLAY_LIMIT]
        print("表示：Edition登録件数が多いWork（一部）。")

print("\n【表示する作品】")
df_plot = pd.DataFrame(graph_rows, columns=["work_id", "title", "linked_edition_records", "year"])
if df_plot.empty:
    print("表示対象はありません。")
else:
    display(df_plot.rename(columns={"work_id":"Work ID", "title":"タイトル",
        "linked_edition_records":"Edition登録件数", "year":"初出・初刊年の目安（Conradの例のみ）"}).reset_index(drop=True))

## 3. Edition登録件数をグラフで見る
棒グラフの1本が、Open Library上の一つの**Workレコード**に対応します。Conradの8作品は文学史上の初出・初刊年の目安で並べています。Edition登録件数は、それぞれの作品が刊行された時期の出版数ではありません。

In [ ]:
# @title Edition登録件数をグラフで表示する {display-mode: "form"}
if df_plot.empty:
    print("表示対象の作品がありません。")
else:
    df = df_plot.iloc[::-1].copy()
    special = AUTHOR_ID.strip() == "OL19441A" and (not YEAR_FROM.strip() or _is_saved)
    labels = [
        (f'{int(x["year"])}  {x["title"]}' if special and pd.notna(x.get("year"))
         else f'{x["title"]} [{x["work_id"]}]')
        for _, x in df.iterrows()
    ]
    fig, ax = plt.subplots(figsize=(11, max(3.5, len(df)*0.57+1)))
    bars = ax.barh(labels, df["linked_edition_records"])
    ax.bar_label(bars, padding=3)
    ax.set_xlabel("Open Library：Edition登録件数")
    ax.set_title("Joseph Conrad：8つのWork" if special else "Work別のEdition登録件数")
    ax.set_xlim(0, max(df["linked_edition_records"])*1.15 if len(df) else 1)
    fig.tight_layout()
    plt.show()

---
**数値の読み方**　Edition登録件数は、出版部数・読者数・文学的評価ではありません。同じ文学作品が複数のWorkに分かれて登録されていることもあります。APIから取得する順位・中央値は、Open Libraryの**Workレコード単位**の値です。

**使用データ**　自由検索：Open LibraryのWorks・Authors・Editionsの2026年2月28日版を用いた講習会用検索DB（未監査のレコードを含みます）。接続できない場合の保存済み8作品：Open Library公開ページの表示件数（2026年10月14日確認）。この二つは時点が異なるため、件数が一致しない場合があります。

**タイトル表記**　*The Nigger of the “Narcissus”* は当時の原題を使用していますが、現在では差別的とされる語を含みます。

[Open Library：Joseph Conrad](https://openlibrary.org/authors/OL19441A) · [*Heart of Darkness*](https://openlibrary.org/works/OL38663W)